In [1]:
import torch
from torch import nn 
from torch.nn import functional as F

In [ ]:
def get_compute_device():
    if torch.cuda.is_available():
        # checks if cuda is available
        return "cuda"
    elif torch.backends.mps.is_available():
        # checks if mps is available (mps is apple's cuda)
        return "mps"
    else:
        # sticking with cpu
        return "cpu"

device = get_compute_device()

In [3]:
torch.set_default_tensor_type(torch.BFloat16Tensor) # reducing the size to fit to memory


/Users/eyash.p24/Code/Machine Learning/llm-experiments/.venv/lib/python3.12/site-packages/torch/__init__.py:1700: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/tensor/python_tensor.cpp:436.)
  _C._set_default_tensor_type(t)


In [22]:
DIM = 4096
FFN_DIM = 14336
N_LAYERS = 32
N_HEADS = 32
N_KV_HEADS = 8
VOCAB_SIZE = 128256
NORM_EPS = 1e-05
ROPE_THETA = 50000
MAX_BATCH_SIZE = 4
MAX_SEQ_LEN = 128
N_KV_HEAD_REP = N_HEADS // N_KV_HEADS
HEAD_DIM = DIM // N_HEADS

In [5]:
# Applying pre-normalization using RMSNorm (llama 2)

class RMSNorm(torch.nn.Module):
    def __init__(self, dim, norm_eps):
        super().__init__()
        self.norm_eps = norm_eps
        self.weight = nn.Parameter(torch.ones(dim))

    def _norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.norm_eps)

    def forward(self, x):
        out = self._norm(x.float()).type_as(x)
        return out * self.weight # (2, 8, dim)

dummy_inp = torch.randn(2, 8, DIM)
norm = RMSNorm(DIM, NORM_EPS)
output = norm(dummy_inp)
print(output.shape)

torch.Size([2, 8, 4096])


In [6]:
1.0 / (ROPE_THETA * torch.arange(0, DIM, 2)[: (DIM // 2)].float() / DIM )

tensor([       inf, 4.0960e-02, 2.0480e-02,  ..., 2.0029e-05, 2.0020e-05,
        2.0010e-05], dtype=torch.float32)

In [7]:
def precompute_freqs_cis(dim , end, theta = 10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(end, device=freqs.device, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs) # complex64
    return freqs_cis

def reshape_for_broadcast(freqs_cis, x):
    ndim = x.ndim
    assert 0 <= 1 < ndim
    assert freqs_cis.shape == (x.shape[1], x.shape[-1])
    shape = [d if i == 1 or i ==ndim -1 else 1 for i, d in enumerate(x.shape)]
    return freqs_cis.view(*shape)

def apply_rotary_emb(xq, xk, freqs_cis):
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    freqs_cis = reshape_for_broadcast(freqs_cis, xq_)
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
    return xq_out.type_as(xq), xk_out.type_as(xk)

dummy_inp1 = torch.randn(2, 8, N_HEADS, HEAD_DIM)
dummy_inp2 = torch.randn(2, 8, N_KV_HEADS, HEAD_DIM)

dummy_freqs_cis = precompute_freqs_cis(HEAD_DIM, MAX_SEQ_LEN*2, ROPE_THETA)
dummy_freqs_cis = dummy_freqs_cis[0 : 0 + 8]

out1, out2 = apply_rotary_emb(dummy_inp1, dummy_inp2, dummy_freqs_cis)
print("-"*30)
print(out1.shape)
print(out2.shape)


------------------------------
torch.Size([2, 8, 32, 128])
torch.Size([2, 8, 8, 128])


In [10]:
class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()

        self.w1 = nn.Linear(DIM, FFN_DIM, bias=False)
        self.w3 = nn.Linear(DIM, FFN_DIM, bias=False)
        self.w2 = nn.Linear(FFN_DIM, DIM, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x) * self.w3(x))) # (2, 8, DIM) = (batch, seq_len, DIM) uses SwiGlU activation function


dummy_inp = torch.randn(2,8, DIM)
feed_forward = FeedForward()
output = feed_forward(dummy_inp)
del feed_forward
print(output.shape)

torch.Size([2, 8, 4096])


In [19]:
# Group Query Attention with KV-cache
class Attention(nn.Module):
    def __init__(self):
        super().__init__()
        self.wq = nn.Linear(DIM, N_HEADS * HEAD_DIM, bias=False)
        self.wk = nn.Linear(DIM, N_KV_HEADS * HEAD_DIM, bias=False)
        self.wv = nn.Linear(DIM, N_KV_HEADS * HEAD_DIM, bias=False)
        self.wo = nn.Linear(N_HEADS * HEAD_DIM, DIM, bias=False) # Weight matrix defined in the MultiheadAttention formula.

        # Create empty caches for keys and values.
        self.cache_k = torch.zeros(
            (
                MAX_BATCH_SIZE,
                MAX_SEQ_LEN,
                N_KV_HEADS,
                HEAD_DIM,
            )
        )
        self.cache_v = torch.zeros(
            (
                MAX_BATCH_SIZE,
                MAX_SEQ_LEN,
                N_KV_HEADS,
                HEAD_DIM,
            )
        )

    def forward(self, x, start_pos, freqs_cis, mask):
        queries, keys, values = self.wq(x), self.wk(x), self.wv(x) 
        
        # q -> (batch_size, seq_len, N_HEADS*HEAD_DIM) | k,v -> (batch_size, seq_len, N_KV_HEADS*HEAD_DIM)
        batch_size, seq_len, _ = x.shape 

        queries = queries.view(batch_size, seq_len, N_HEADS, HEAD_DIM)
        keys = keys.view(batch_size, seq_len, N_KV_HEADS, HEAD_DIM)
        values = values.view(batch_size, seq_len, N_KV_HEADS, HEAD_DIM)

        queries, keys = apply_rotary_emb(queries, keys, freqs_cis=freqs_cis)

        self.cache_k = self.cache_k.to(queries.device)
        self.cache_v = self.cache_v.to(queries.device)

        self.cache_k[: batch_size, start_pos: start_pos + seq_len] = keys
        self.cache_v[: batch_size, start_pos: start_pos + seq_len] = values

        keys = self.cache_k[:batch_size, :start_pos + seq_len]
        values = self.cache_v[:batch_size, :start_pos + seq_len]

        keys = torch.repeat_interleave(
            keys, dim=2, repeats=N_KV_HEAD_REP
        ) # (batch_size, seq_len, N_KV_HEADS, HEAD_DIM) -> (batch_size, seq_len, N_HEADS, HEAD_DIM)

        values = torch.repeat_interleave(
            values, dim=2, repeats=N_KV_HEAD_REP
        ) # (batch_size, seq_len, N_KV_HEADS, HEAD_DIM) -> (batch_size, seq_len, N_HEADS, HEAD_DIM)

        # reshaping for scaled_dot_product_attention (batch_size, ..., seq_len, HEAD_DIM)
        queries = queries.transpose(1,2) # (batch, seq_len, N_HEADS, HEAD_DIM) -> (batch_size, N_HEADS, seq_len, HEAD_DIM)
        keys = keys.transpose(1,2) # (batch, seq_len, N_HEADS, HEAD_DIM) -> (batch_size, N_HEADS, seq_len, HEAD_DIM)
        values = values.transpose(1,2) # (batch, seq_len, N_HEADS, HEAD_DIM) -> (batch_size, N_HEADS, seq_len, HEAD_DIM)

        out = F.scaled_dot_product_attention(
            queries,
            keys,
            values,
            attn_mask=mask
        ) # (batch_size, N_HEADS, seq_len, HEAD_DIM)

        # transpose, (batch_size, seq_len, N_HEADS, HEAD_DIM) - (batch_size, seq_len, DIM) - -1 does N_HEAD * HEAD_DIM = DIM
        out = out.transpose(1,2).contiguous().view(batch_size, seq_len, -1)
        return self.wo(out) # (batch_size, seq_len, DIM)


dummy_inp = torch.randn(2,8,DIM)
dummy_start_pos = 0
dummy_freqs_cis = torch.randn(8, 64)
dummy_mask = torch.randn(8, 8)

attention = Attention()
output = attention(dummy_inp, dummy_start_pos, dummy_freqs_cis, dummy_mask)
print(output.shape)

del attention

torch.Size([2, 8, 4096])


In [21]:
class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()

        self.attention = Attention()
        self.feed_forward = FeedForward()
        self.attention_norm = RMSNorm(DIM, NORM_EPS)
        self.ffn_norm = RMSNorm(DIM, NORM_EPS)

    def forward(self, x, start_pos, freqs_cis, mask):
        h = x + self.attention(
            self.attention_norm(x),
            start_pos,
            freqs_cis,
            mask
        ) # (2, 8, 4096) = (batch_size, seq_len, DIM)

        out = h + self.feed_forward(
            self.ffn_norm(h)
        ) # (2, 8, 4096) = (batch_size, seq_len, DIM)

        return out

dummy_inp = torch.randn(2, 8, DIM)
dummy_start_pos = 0
dummy_freqs_cis = torch.randn(8, 64)
dummy_mask = torch.randn(8, 8)

transformer_block = TransformerBlock()

output = transformer_block(dummy_inp, dummy_start_pos, dummy_freqs_cis, dummy_mask)

print(output.shape) # (2, 8, DIM) = (batch_size, seq_len, DIM)
    

torch.Size([2, 8, 4096])


In [27]:
class Transformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_embedding = nn.Embedding(
            VOCAB_SIZE, DIM
        )
        self.layers = torch.nn.ModuleList()
        for _ in range(N_LAYERS):
            self.layers.append(TransformerBlock())

        self.norm = RMSNorm(DIM, NORM_EPS)
        self.output = nn.Linear(DIM, VOCAB_SIZE, bias=False)

        self.freq_cis = precompute_freqs_cis(
            HEAD_DIM,
            MAX_SEQ_LEN * 2,
            ROPE_THETA,
        )

    @torch.inference_mode()
    def forward(self, tokens, start_pos):
        _batch_size, seq_len = tokens.shape
        h = self.tok_embedding(tokens)
        self.freq_cis = self.freq_cis.to(tokens.device)
        freq_cis = self.freq_cis[start_pos: start_pos + seq_len]

        # When we take the tokens from the cached values (seqlen=1) we don't need any aditional mask.
        mask = None

        # Because of KV Cache, we process only 1 token. However, the first run doesn't have any cache. So it has a seqlen > 1.
        if seq_len > 1:
            # Since this is the first pass, we don't have any KV Cache. So we need a mask. Create (seqlen, seqlen) matrix with float("-inf") values.
            mask = torch.full((seq_len, seq_len), float("-inf"), device=tokens.device)
            
            # Take the upper triangle excluding diagonal since it's casual LM.
            mask = torch.triu(mask, diagonal=1).to(tokens.device)

        for layer in self.layers:
            h = layer(h, start_pos, freq_cis, mask) # (2, 8, 4096) = (batch_size, seq_len, DIM)

        h = self.norm(h) # (2, 8, 4096) = (batch_size, seq_len, DIM)

        out = self.output(h).float() # (2, 8, 128256) = (batch_size, seq_len, VOCAB_SIZE)
        return out


# nn.Embeddings only accepts positive number
dummy_tokens = torch.rand(2, 8).long()
dummy_start_pos = 0
transformer = Transformer()
output = transformer(dummy_tokens, dummy_start_pos)
print(output.shape) # (2, 8, 128256) = (bsz, seqlen, VOCAB_SIZE)

del transformer

torch.Size([2, 8, 128256])


In [28]:
torch.mps.empty_cache()